## Section 1 · Imports & Load Preprocessed Data

In [ ]:
import os, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
BASE_DIR = Path('/kaggle/input/notebooks/esraa320230011/preprocessing-f')
SPEC_DIR = BASE_DIR / 'specs'

print('Files available in input:')
for f in sorted(BASE_DIR.iterdir()):
    size = f.stat().st_size / 1e6 if f.is_file() else 0
    tag  = f'({size:.1f} MB)' if f.is_file() else '(folder)'
    print(f'  {f.name:40s}  {tag}')

npy_files = sorted(SPEC_DIR.glob('*.npy'))
print(f'\nSpec files found: {len(npy_files)}')
if len(npy_files) == 0:
    raise FileNotFoundError(f'No .npy files found in {SPEC_DIR}.')


# Load config
with open(BASE_DIR / 'config.json') as f:
    CFG = json.load(f)

CFG['epochs']   = 50
CFG['patience'] = 10
CFG['bs']       = 32

print('Config loaded:')
for k, v in CFG.items():
    print(f'  {k}: {v}')


# Load manifest
manifest = pd.read_csv(BASE_DIR / 'manifest.csv')
print(f'Manifest rows    : {len(manifest)}')
print(f'Manifest columns : {list(manifest.columns)}')

# Detect label column
label_col = None
for col in manifest.columns:
    if any(kw in col.lower() for kw in ['label', 'class', 'target']):
        label_col = col; break
if label_col is None:
    raise ValueError('No label column found in manifest.csv')
print(f'Label column     : {label_col}')

manifest['path'] = manifest['path'].apply(
    lambda p: str(SPEC_DIR / Path(p).name)
)

exists_mask = manifest['path'].apply(lambda p: Path(p).exists())
print(f'Files on disk    : {exists_mask.sum()} / {len(manifest)}')
manifest = manifest[exists_mask].reset_index(drop=True)

print(f'\nClass distribution:')
print(manifest[label_col].value_counts().rename({0:'Interictal', 1:'Ictal'}))
print(f'Seizure %: {100*manifest[label_col].mean():.2f}%')

sample_spec = np.load(manifest['path'].iloc[0])
print(f'Single spec shape: {sample_spec.shape}')  # (n_ch, F, T)
n_ch_spec, F_bins, T_frames = sample_spec.shape
print(f'  Channels={n_ch_spec}  FreqBins={F_bins}  TimeFrames={T_frames}')

## Section 2 · Dataset & DataLoaders

In [ ]:
class EEGManifestDataset(Dataset):
    def __init__(self, df, label_col, augment=False):
        self.paths   = df['path'].values
        self.labels  = df[label_col].values.astype(np.int64)
        self.augment = augment

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        spec  = torch.from_numpy(np.load(self.paths[idx])).float()  # (C, F, T)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.augment:
            spec = self._spec_augment(spec)
        return spec, label

    @staticmethod
    def _spec_augment(x, freq_param=8, time_param=8):
        x = x.clone()
        _, F, T = x.shape
        f  = torch.randint(0, freq_param+1, (1,)).item()
        f0 = torch.randint(0, max(F-f, 1), (1,)).item()
        x[:, f0:f0+f, :] = 0.0
        t  = torch.randint(0, time_param+1, (1,)).item()
        t0 = torch.randint(0, max(T-t, 1), (1,)).item()
        x[:, :, t0:t0+t] = 0.0
        return x


train_df, tmp_df = train_test_split(
    manifest, test_size=0.30, stratify=manifest[label_col], random_state=SEED)
val_df,  test_df = train_test_split(
    tmp_df,   test_size=0.50, stratify=tmp_df[label_col],   random_state=SEED)

print(f'Train: {len(train_df):5d}  Ictal={train_df[label_col].mean():.3f}')
print(f'Val  : {len(val_df):5d}  Ictal={val_df[label_col].mean():.3f}')
print(f'Test : {len(test_df):5d}  Ictal={test_df[label_col].mean():.3f}')

y_tr    = train_df[label_col].values
counts  = np.bincount(y_tr)
weights = (1.0 / counts)[y_tr]
sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(y_tr), replacement=True)

BS = CFG['bs']
train_loader = DataLoader(EEGManifestDataset(train_df, label_col, augment=True),
                          batch_size=BS, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(EEGManifestDataset(val_df,   label_col), batch_size=BS,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(EEGManifestDataset(test_df,  label_col), batch_size=BS,
                          shuffle=False, num_workers=2, pin_memory=True)

spec_b, lab_b = next(iter(train_loader))
print(f'\nBatch shape: {spec_b.shape}  labels: {lab_b[:8].tolist()}')

## Section 3 · RNN Architecture (Bidirectional GRU + Temporal Attention)

In [ ]:
# Bidirectional GRU (RNN) Model
class EEG_RNN(nn.Module):
    """
    Input : (B, n_ch, n_samples)  — raw EEG sequence
    Output: (B, 2)  logits

    Architecture:
      Flatten (C, F) -> feature_dim per timestep
      Linear projection -> LayerNorm -> ReLU
      3-layer Bidirectional GRU
      Last hidden state -> BN -> Dropout -> FC -> 2
    """
    def __init__(self, in_ch, F_bins, T_frames,
                 hidden=128, n_layers=3, dropout=0.4, n_classes=2):
        super().__init__()
        feat_dim = in_ch * F_bins   # flattened C×F per time step

        self.proj = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(inplace=True),
        )
        self.rnn = nn.GRU(
            input_size=hidden,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        B, C, F, T = x.shape
        # Reshape to (B, T, C*F)  — T is the sequence axis
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)
        x = self.proj(x)                    # (B, T, hidden)
        out, _ = self.rnn(x)                # (B, T, hidden*2)
        return self.head(out[:, -1, :])     # last time step → (B, n_classes)


model = EEG_RNN(in_ch=n_ch_spec, F_bins=F_bins, T_frames=T_frames).to(DEVICE)
out   = model(spec_b.to(DEVICE))
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Output shape     : {out.shape}  (expected B×2)')
print(f'Trainable params : {total:,}')

## Section 4 · Loss, Optimizers & Scheduler

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        a   = torch.where(targets==1,
                          torch.full_like(ce, self.alpha),
                          torch.full_like(ce, 1-self.alpha))
        return (a * (1-p_t)**self.gamma * ce).mean()


OPTIMIZER_NAME = 'adamw'   # 'adam' | 'adamw' | 'nadam' | 'adamax' | 'sgd_mom'
LR = 1e-3
WD = 1e-4

def get_optimizer(name, model):
    p = model.parameters()
    if name == 'adam'   : return Adam(p, lr=LR, weight_decay=WD)
    if name == 'adamw'  : return AdamW(p, lr=LR, weight_decay=WD, betas=(0.9,0.999))
    if name == 'nadam'  : return torch.optim.NAdam(p, lr=LR, weight_decay=WD)
    if name == 'adamax' : return torch.optim.Adamax(p, lr=LR, weight_decay=WD)
    if name == 'sgd_mom': return torch.optim.SGD(p, lr=LR, momentum=0.9,
                                                  weight_decay=WD, nesterov=True)
    raise ValueError(name)

optimizer = get_optimizer(OPTIMIZER_NAME, model)
criterion = FocalLoss(gamma=2.0, alpha=0.75)
# CosineAnnealingWarmRestarts works well for RNNs (periodic restarts escape local minima)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
print(f'Optimizer : {OPTIMIZER_NAME.upper()} | LR={LR} | WD={WD}')

## Section 5 · Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = correct = total = 0
    for spec_b, lab_b in loader:
        spec_b, lab_b = spec_b.to(DEVICE), lab_b.to(DEVICE)
        logits = model(spec_b)
        loss   = criterion(logits, lab_b)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * lab_b.size(0)
        correct    += (logits.argmax(1) == lab_b).sum().item()
        total      += lab_b.size(0)
    return total_loss/total, correct/total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []
    for spec_b, lab_b in loader:
        spec_b, lab_b = spec_b.to(DEVICE), lab_b.to(DEVICE)
        logits = model(spec_b)
        loss   = criterion(logits, lab_b)
        total_loss += loss.item() * lab_b.size(0)
        correct    += (logits.argmax(1) == lab_b).sum().item()
        total      += lab_b.size(0)
        all_probs.extend(torch.softmax(logits,1)[:,1].cpu().numpy())
        all_labels.extend(lab_b.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels))>1 else 0.0
    return total_loss/total, correct/total, auc, all_probs, all_labels


history    = defaultdict(list)
best_auc   = 0.0
patience_c = 0
best_state = None

print(f'Training RNN (Bi-GRU / {OPTIMIZER_NAME.upper()}) for up to {CFG["epochs"]} epochs ...\n')
for epoch in range(1, CFG['epochs']+1):
    tr_loss, tr_acc              = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc, va_auc,_,_ = evaluate(model, val_loader, criterion)
    scheduler.step(epoch)

    history['train_loss'].append(tr_loss);  history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);    history['val_acc'].append(va_acc)
    history['val_auc'].append(va_auc)

    flag = ''
    if va_auc > best_auc:
        best_auc   = va_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_c = 0; flag = ' ★'
    else:
        patience_c += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'Ep {epoch:3d}/{CFG["epochs"]} '
              f'| tr={tr_loss:.4f}/{tr_acc:.3f} '
              f'| va={va_loss:.4f}/{va_acc:.3f} '
              f'auc={va_auc:.4f}{flag}')

    if patience_c >= CFG['patience']:
        print(f'Early stopping at epoch {epoch}'); break

model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print(f'\nBest val AUC: {best_auc:.4f}')

## Section 6 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes,
    [('train_loss','val_loss'), ('train_acc','val_acc'), ('val_auc',)],
    ['Loss', 'Accuracy', 'Val AUC']):
    for m in metric:
        ax.plot(history[m], label=m)
    ax.set(title=title, xlabel='Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle(f'RNN (Bi-GRU) Training — {OPTIMIZER_NAME.upper()}', y=1.02)
plt.tight_layout(); plt.show()

## Section 7 · Test Evaluation

In [ ]:
_, acc, auc, probs, labels_te = evaluate(model, test_loader, criterion)
preds = (np.array(probs) > 0.5).astype(int)

print(f'Test Accuracy : {acc:.4f}')
print(f'Test AUC      : {auc:.4f}')
print(f'Test F1-macro : {f1_score(labels_te, preds, average="macro"):.4f}')
print('\nClassification Report:')
print(classification_report(labels_te, preds, target_names=['Interictal','Ictal']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(labels_te, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Interictal','Ictal'],
            yticklabels=['Interictal','Ictal'], ax=axes[0])
axes[0].set(title='RNN — Confusion Matrix', xlabel='Predicted', ylabel='True')
fpr, tpr, _ = roc_curve(labels_te, probs)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC={auc:.4f}')
axes[1].plot([0,1],[0,1],'--', color='gray')
axes[1].set(title='RNN — ROC Curve', xlabel='FPR', ylabel='TPR')
axes[1].legend()
plt.tight_layout(); plt.show()

## Section 8 · Save Model

In [ ]:
out_dir = Path('/kaggle/working')
torch.save(model.state_dict(), out_dir / 'rnn_best.pt')
print(f'Saved → {out_dir}/rnn_best.pt')